# 03 — Model Comparison

This notebook trains and compares multiple classifiers on the 90-feature union set selected in notebook 02. All models use `class_weight='balanced'` to handle the severe class imbalance (class 5 = 45% of data).

**Models compared:**
- Logistic Regression
- Random Forest
- XGBoost
- MLP Neural Network
- Voting Classifier (LR + RF + XGB, soft voting)
- Stacking Classifier (LR + RF meta-learner)
- Calibrated SVM
- AdaBoost / Extra Trees

**Evaluation metric:** Weighted Log Loss (penalises errors on rare classes more heavily)

## Setup — Load Data, Scale, Select Features

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# load featues
X_train = pd.read_csv("../raw datasets/X_train.csv")
y_train = pd.read_csv("../raw datasets/y_train.csv")

# train features
x_train, x_test, y_train_split, y_test_split = train_test_split(
    X_train, y_train,
    test_size=0.2,
    stratify=y_train, # ensure class distribution is preserved
    random_state=42 #ensures that the train/test split is reproducible every time you run your code.
)
# flattens dataset scikit-learn functions expect a 1D array for classification targets, like (8000,)
y_train_split = y_train_split.values.ravel()
y_test_split = y_test_split.values.ravel()


# Fit scaler on training data
scaler = StandardScaler()
scaler.fit(x_train)

# Transform without overwriting yet
x_train_scaled = pd.DataFrame(
    scaler.transform(x_train),
    columns=X_train.columns,
    index=x_train.index
)

x_test_scaled = pd.DataFrame(
    scaler.transform(x_test),
    columns=X_train.columns,
    index=x_test.index
)

# Convert scaled arrays back into DataFrames
x_train = pd.DataFrame(x_train_scaled, columns=X_train.columns, index=x_train.index)
x_test = pd.DataFrame(x_test_scaled, columns=X_train.columns, index=x_test.index)

selected_features = [2, 9, 14, 17, 19, 32, 39, 41, 44, 46, 47, 48, 53, 55, 59, 61, 64, 65, 66, 67, 71, 79, 85, 86, 87, 88, 
                     89, 90, 95, 97, 99, 100, 105, 106, 111, 112, 123, 124, 127, 130, 131, 132, 135, 140, 141, 148, 152, 
                     154, 157, 164, 168, 172, 176, 177, 182, 187, 198, 200, 201, 205, 206, 215, 216, 218, 222, 228, 231, 
                     236, 240, 242, 249, 259, 263, 265, 267, 268, 270, 273, 274, 279, 282, 283, 284, 287, 289, 290, 292, 293]

## Evaluation Helper Function

All models use the same `evaluate_model()` wrapper which computes weighted log loss on both train and validation sets.

In [2]:
from sklearn.preprocessing import label_binarize
import numpy as np

def evaluate_model(model, model_name="Model"):
    """Train model on selected features and report weighted log loss on train and validation sets."""
    x_train_k = x_train_scaled.iloc[:, selected_features]
    x_test_k = x_test_scaled.iloc[:, selected_features]

    model.fit(x_train_k, y_train_split)

    y_pred_train = model.predict_proba(x_train_k)
    y_pred_test = model.predict_proba(x_test_k)

    classes = np.unique(y_train_split)
    y_train_ohe = label_binarize(y_train_split, classes=classes)
    y_test_ohe = label_binarize(y_test_split, classes=classes)

    def weighted_log_loss(y_true, y_pred):
        class_counts = np.sum(y_true, axis=0)
        class_weights = 1.0 / class_counts
        class_weights /= np.sum(class_weights)
        sample_weights = np.sum(y_true * class_weights, axis=1)
        return -np.mean(sample_weights * np.sum(y_true * np.log(y_pred + 1e-15), axis=1))

    train_loss = weighted_log_loss(y_train_ohe, y_pred_train)
    test_loss = weighted_log_loss(y_test_ohe, y_pred_test)

    print(f"\n📊 {model_name}")
    print(f"Weighted Log Loss on Train Set: {train_loss:.4f}")
    print(f"Weighted Log Loss on Test Set : {test_loss:.4f}")

## Logistic Regression

In [3]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000, class_weight='balanced')
evaluate_model(lr, "Logistic Regression")


📊 Logistic Regression
Weighted Log Loss on Train Set: 0.0018
Weighted Log Loss on Test Set : 0.0070


## Random Forest

In [4]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
evaluate_model(rf, "Random Forest")


📊 Random Forest
Weighted Log Loss on Train Set: 0.0013
Weighted Log Loss on Test Set : 0.0175


## XGBoost

In [5]:
from xgboost import XGBClassifier
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, eval_metric='mlogloss', random_state=42)
evaluate_model(xgb, "XGBoost")


📊 XGBoost
Weighted Log Loss on Train Set: 0.0003
Weighted Log Loss on Test Set : 0.0079


## MLP Neural Network

In [6]:
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)
evaluate_model(mlp, "MLP Neural Network")


📊 MLP Neural Network
Weighted Log Loss on Train Set: 0.0000
Weighted Log Loss on Test Set : 0.0216


c:\Users\huiqi\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


## Voting Classifier (Soft — LR + RF + XGB)

In [7]:
from sklearn.ensemble import VotingClassifier
voting = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
        ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
        ('xgb', XGBClassifier(n_estimators=100, eval_metric='mlogloss', random_state=42))
    ],
    voting='soft'
)
evaluate_model(voting, "Voting Classifier (Soft)")


📊 Voting Classifier (Soft)
Weighted Log Loss on Train Set: 0.0008
Weighted Log Loss on Test Set : 0.0061


## Stacking Classifier (LR + RF → LR meta-learner)

Best performing ensemble: WLL train=0.0032, WLL val=0.0045

In [8]:
from sklearn.ensemble import StackingClassifier
stacked = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
        ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
    ],
    final_estimator=LogisticRegression(max_iter=1000, class_weight='balanced'),
    cv=5
)
evaluate_model(stacked, "Stacking Classifier (LR + RF)")


📊 Stacking Classifier (LR + RF)
Weighted Log Loss on Train Set: 0.0032
Weighted Log Loss on Test Set : 0.0045


## Calibrated SVM

In [9]:
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
svc = SVC(kernel='rbf', probability=True)
cal_svc = CalibratedClassifierCV(svc, cv=3)
evaluate_model(cal_svc, "Calibrated SVM")


📊 Calibrated SVM
Weighted Log Loss on Train Set: 0.0048
Weighted Log Loss on Test Set : 0.0072


## AdaBoost & Extra Trees

In [10]:
from sklearn.ensemble import AdaBoostClassifier, ExtraTreesClassifier
evaluate_model(AdaBoostClassifier(n_estimators=100, random_state=42), "AdaBoost")
evaluate_model(ExtraTreesClassifier(n_estimators=100, random_state=42), "Extra Trees")


📊 AdaBoost
Weighted Log Loss on Train Set: 0.0110
Weighted Log Loss on Test Set : 0.0086

📊 Extra Trees
Weighted Log Loss on Train Set: -0.0000
Weighted Log Loss on Test Set : 0.0180


## Results Summary

| Model | WLL Train | WLL Validation |
|---|---|---|
| Logistic Regression | — | — |
| Random Forest | 0.0013 | 0.0153 |
| XGBoost | 0.0001 | 0.0078 |
| MLP Neural Network | 0.0000 | 0.0216 |
| Voting Classifier | 0.0008 | 0.0061 |
| **Stacking (LR + RF)** | **0.0032** | **0.0045** |
| Calibrated SVM | 0.0048 | 0.0072 |
| AdaBoost | 0.0108 | 0.0112 |
| Extra Trees | ~0.0000 | 0.0180 |

**Selected: Stacking Classifier** — best balance of train/validation WLL with least overfitting.

## Evaluation on Test Set 2 (Distribution Shift)

Test Set 2 has covariate shift — feature distributions differ from training. We evaluate the stacked model on the 202 labelled samples from Test Set 2.

In [11]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Load datasets
x_train = pd.read_csv("../raw datasets/X_train.csv")
y_train = pd.read_csv("../raw datasets/y_train.csv").values.ravel()  # Flatten to 1D

x_test_2 = pd.read_csv("../raw datasets/X_test_2.csv")
y_test_2 = pd.read_csv("../raw datasets/y_test_2_reduced.csv").values.ravel()

# Fit scaler only on training data
scaler = StandardScaler()
x_train_scaled = pd.DataFrame(
    scaler.fit_transform(x_train),
    columns=x_train.columns,
    index=x_train.index
)

x_test_2_scaled = pd.DataFrame(
    scaler.transform(x_test_2),
    columns=x_test_2.columns,
    index=x_test_2.index
)

# Define selected features by index
selected_features = [2, 9, 10, 14, 17, 19, 20, 32, 39, 41, 44, 46, 47, 48, 50, 53, 55, 59, 61, 64, 65, 66, 67, 70, 71,
                     79, 82, 85, 86, 87, 88, 89, 90, 95, 97, 99, 100, 105, 106, 111, 112, 113, 116, 118, 123, 124, 125,
                     127, 130, 131, 132, 134, 135, 140, 141, 144, 148, 152, 154, 157, 162, 164, 168, 172, 176, 177, 182,
                     187, 198, 200, 201, 205, 206, 215, 216, 218, 222, 227, 228, 230, 231, 236, 240, 242, 249, 257, 259,
                     263, 265, 267, 268, 270, 271, 273, 274, 276, 279, 282, 283, 284, 287, 289, 290, 292, 293]

# Subset only the selected features
x_train_final = x_train_scaled.iloc[:, selected_features]
x_test_2_final = x_test_2_scaled.iloc[:, selected_features]

In [16]:
from sklearn.metrics import log_loss
from sklearn.preprocessing import label_binarize
import numpy as np

def evaluate_model_final(model, model_name="Model"):
    x_train_k = x_train_scaled.iloc[:, selected_features]
    x_test_k = x_test_2_scaled.iloc[:, selected_features]

    model.fit(x_train_k, y_train)

    y_pred_train = model.predict_proba(x_train_k)
    
    # y_test_2 only has 202 labels — slice x_test to match
    x_test_labelled = x_test_k.iloc[:len(y_test_2)]
    y_pred_test = model.predict_proba(x_test_labelled)

    classes = np.unique(y_train)
    y_train_ohe = label_binarize(y_train, classes=classes)
    y_test_ohe = label_binarize(y_test_2, classes=classes)

    def weighted_log_loss(y_true_ohe, y_pred):
        class_counts = np.sum(y_true_ohe, axis=0)
        class_counts = np.where(class_counts == 0, 1, class_counts)
        class_weights = 1.0 / class_counts
        class_weights /= np.sum(class_weights)
        sample_weights = np.sum(y_true_ohe * class_weights, axis=1)
        return np.mean(-sample_weights * np.sum(y_true_ohe * np.log(y_pred + 1e-15), axis=1))

    train_loss = weighted_log_loss(y_train_ohe, y_pred_train)
    test_loss = weighted_log_loss(y_test_ohe, y_pred_test)

    print(f"\n📊 {model_name}")
    print(f"Weighted Log Loss on Train Set: {train_loss:.4f}")
    print(f"Weighted Log Loss on Test Set : {test_loss:.4f}")

In [17]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

stacked_final = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
        ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
        ('xgb', XGBClassifier(n_estimators=100, eval_metric='mlogloss', random_state=42))
    ],
    final_estimator=LogisticRegression(max_iter=1000, class_weight='balanced'),
    cv=5
)
evaluate_model_final(stacked_final, "Final Stacked Model on Test Set 2")


📊 Final Stacked Model on Test Set 2
Weighted Log Loss on Train Set: 0.0037
Weighted Log Loss on Test Set : 0.0147
